# In-Context Learning Notebook

This notebook implements training for in-context learning models, based on the implementation in `train.py`.

## Environment Setup

First, let's check if we're running in Colab and set up the environment accordingly.

In [1]:
import os
import sys

# Check if we're running in Google Colab
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

# Add src to path
if IN_COLAB:
    # Install required packages for Colab
    !pip install -q funcy tqdm torch transformers wandb xgboost scikit-learn pyyaml

    # Use Colab's built-in content root
    sys.path.append('/content')
else:
    # If running locally, add src to path
    sys.path.append('src')

Running in Google Colab: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.8 MB/s eta 0:00:00


## Check GPU Availability

Let's check if a GPU is available for training.

In [2]:
import torch

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("No GPU available, using CPU for training.")

# Enable cuDNN benchmarking for better performance
torch.backends.cudnn.benchmark = True

Torch version: 2.6.0+cu124
CUDA available: True
CUDA device: Tesla T4
CUDA version: 12.4


import os
import uuid
import yaml
from random import randint
from tqdm.notebook import tqdm
import torch
import matplotlib.pyplot as plt
import numpy as np

In [3]:
# Schema
# Define constants and schema for the configuration

# Task List
TASK_LIST = [
    "linear_regression",
    "sparse_linear_regression",
    "linear_classification",
    "relu_2nn_regression",
    "decision_tree",
]

# Model Schema
model_schema = {
    "family": ["gpt2", "lstm"],
    "n_positions": None,  # maximum context length
    "n_dims": None,  # latent dimension
    "n_embd": None,
    "n_layer": None,
    "n_head": None,
}

# Curriculum Schema
curriculum_base_schema = {
    "start": None,  # initial parameter
    "end": None,  # limit of final value
    "inc": None,  # how much to increment each time
    "interval": None,  # increment every how many steps
}

curriculum_schema = {
    "dims": curriculum_base_schema,
    "points": curriculum_base_schema,
}

# Training Schema
training_schema = {
    "task": TASK_LIST,
    "task_kwargs": {},
    "num_tasks": None,
    "num_training_examples": None,
    "data": ["gaussian"],
    "batch_size": 64,
    "learning_rate": 3e-4,
    "train_steps": 1000,
    "save_every_steps": 1000,  # how often to checkpoint
    "keep_every_steps": -1,  # permanent checkpoints
    "resume_id": None,  # run uuid64
    "curriculum": curriculum_schema,
}

# Wandb Schema
wandb_schema = {
    "project": "in-context-training",
    "entity": "in-context",
    "notes": "",
    "name": None,
    "log_every_steps": 10,
}

# Complete Schema
schema = {
    "out_dir": None,
    "model": model_schema,
    "training": training_schema,
    "wandb": wandb_schema,
    "test_run": False,
}

In [4]:
# Base Models
# Neural network implementations for models.py

class NeuralNetwork(torch.nn.Module):
    def __init__(self, in_size=50, hidden_size=1000, out_size=1):
        super(NeuralNetwork, self).__init__()

        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_size, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_size, out_size),
        )

    def forward(self, x):
        out = self.net(x)
        return out


class ParallelNetworks(torch.nn.Module):
    def __init__(self, num_models, model_class, **model_class_init_args):
        super(ParallelNetworks, self).__init__()
        self.nets = torch.nn.ModuleList(
            [model_class(**model_class_init_args) for i in range(num_models)]
        )

    def forward(self, xs):
        assert xs.shape[0] == len(self.nets)

        for i in range(len(self.nets)):
            out = self.nets[i](xs[i])
            if i == 0:
                outs = torch.zeros(
                    [len(self.nets)] + list(out.shape), device=out.device
                )
            outs[i] = out
        return outs

In [5]:
# Models
# Main model implementations

from transformers import GPT2Model, GPT2Config
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, Lasso
import warnings
from sklearn import tree
import xgboost as xgb


def build_model(conf):
    if conf.family == "gpt2":
        model = TransformerModel(
            n_dims=conf.n_dims,
            n_positions=conf.n_positions,
            n_embd=conf.n_embd,
            n_layer=conf.n_layer,
            n_head=conf.n_head,
        )
    else:
        raise NotImplementedError

    return model


def get_relevant_baselines(task_name):
    task_to_baselines = {
        "linear_regression": [
            (LeastSquaresModel, {}),
            (NNModel, {"n_neighbors": 3}),
            (AveragingModel, {}),
        ],
        "linear_classification": [
            (NNModel, {"n_neighbors": 3}),
            (AveragingModel, {}),
        ],
        "sparse_linear_regression": [
            (LeastSquaresModel, {}),
            (NNModel, {"n_neighbors": 3}),
            (AveragingModel, {}),
        ]
        + [(LassoModel, {"alpha": alpha}) for alpha in [1, 0.1, 0.01, 0.001, 0.0001]],
        "relu_2nn_regression": [
            (LeastSquaresModel, {}),
            (NNModel, {"n_neighbors": 3}),
            (AveragingModel, {}),
            (
                GDModel,
                {
                    "model_class": NeuralNetwork,
                    "model_class_args": {
                        "in_size": 20,
                        "hidden_size": 100,
                        "out_size": 1,
                    },
                    "opt_alg": "adam",
                    "batch_size": 100,
                    "lr": 5e-3,
                    "num_steps": 100,
                },
            ),
        ],
        "decision_tree": [
            (LeastSquaresModel, {}),
            (NNModel, {"n_neighbors": 3}),
            (DecisionTreeModel, {"max_depth": 4}),
            (DecisionTreeModel, {"max_depth": None}),
            (XGBoostModel, {}),
            (AveragingModel, {}),
        ],
    }

    models = [model_cls(**kwargs) for model_cls, kwargs in task_to_baselines[task_name]]
    return models


class TransformerModel(torch.nn.Module):
    def __init__(self, n_dims, n_positions, n_embd=128, n_layer=12, n_head=4):
        super(TransformerModel, self).__init__()
        configuration = GPT2Config(
            n_positions=2 * n_positions,
            n_embd=n_embd,
            n_layer=n_layer,
            n_head=n_head,
            resid_pdrop=0.0,
            embd_pdrop=0.0,
            attn_pdrop=0.0,
            use_cache=False,
        )
        self.name = f"gpt2_embd={n_embd}_layer={n_layer}_head={n_head}"

        self.n_positions = n_positions
        self.n_dims = n_dims
        self._read_in = torch.nn.Linear(n_dims, n_embd)
        self._backbone = GPT2Model(configuration)
        self._read_out = torch.nn.Linear(n_embd, 1)

    @staticmethod
    def _combine(xs_b, ys_b):
        """Interleaves the x's and the y's into a single sequence."""
        bsize, points, dim = xs_b.shape
        ys_b_wide = torch.cat(
            (
                ys_b.view(bsize, points, 1),
                torch.zeros(bsize, points, dim - 1, device=ys_b.device),
            ),
            axis=2,
        )
        zs = torch.stack((xs_b, ys_b_wide), dim=2)
        zs = zs.view(bsize, 2 * points, dim)
        return zs

    def forward(self, xs, ys, inds=None):
        if inds is None:
            inds = torch.arange(ys.shape[1])
        else:
            inds = torch.tensor(inds)
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")
        zs = self._combine(xs, ys)
        embeds = self._read_in(zs)
        output = self._backbone(inputs_embeds=embeds).last_hidden_state
        prediction = self._read_out(output)
        return prediction[:, ::2, 0][:, inds]  # predict only on xs


class NNModel:
    def __init__(self, n_neighbors, weights="uniform"):
        # should we be picking k optimally
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.name = f"NN_n={n_neighbors}_{weights}"

    def __call__(self, xs, ys, inds=None):
        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []

        for i in inds:
            if i == 0:
                preds.append(torch.zeros_like(ys[:, 0]))  # predict zero for first point
                continue
            train_xs, train_ys = xs[:, :i], ys[:, :i]
            test_x = xs[:, i : i + 1]
            dist = (train_xs - test_x).square().sum(dim=2).sqrt()

            if self.weights == "uniform":
                weights = torch.ones_like(dist)
            else:
                weights = 1.0 / dist
                inf_mask = torch.isinf(weights).float()  # deal with exact match
                inf_row = torch.any(inf_mask, axis=1)
                weights[inf_row] = inf_mask[inf_row]

            pred = []
            k = min(i, self.n_neighbors)
            ranks = dist.argsort()[:, :k]
            for y, w, n in zip(train_ys, weights, ranks):
                y, w = y[n], w[n]
                pred.append((w * y).sum() / w.sum())
            preds.append(torch.stack(pred))

        return torch.stack(preds, dim=1)


# xs and ys should be on cpu for this method. Otherwise the output maybe off in case when train_xs is not full rank due to the implementation of torch.linalg.lstsq.
class LeastSquaresModel:
    def __init__(self, driver=None):
        self.driver = driver
        self.name = f"OLS_driver={driver}"

    def __call__(self, xs, ys, inds=None):
        xs, ys = xs.cpu(), ys.cpu()
        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []

        for i in inds:
            if i == 0:
                preds.append(torch.zeros_like(ys[:, 0]))  # predict zero for first point
                continue
            train_xs, train_ys = xs[:, :i], ys[:, :i]
            test_x = xs[:, i : i + 1]

            ws, _, _, _ = torch.linalg.lstsq(
                train_xs, train_ys.unsqueeze(2), driver=self.driver
            )

            pred = test_x @ ws
            preds.append(pred[:, 0, 0])

        return torch.stack(preds, dim=1)


class AveragingModel:
    def __init__(self):
        self.name = "averaging"

    def __call__(self, xs, ys, inds=None):
        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []

        for i in inds:
            if i == 0:
                preds.append(torch.zeros_like(ys[:, 0]))  # predict zero for first point
                continue
            train_xs, train_ys = xs[:, :i], ys[:, :i]
            test_x = xs[:, i : i + 1]

            train_zs = train_xs * train_ys.unsqueeze(dim=-1)
            w_p = train_zs.mean(dim=1).unsqueeze(dim=-1)
            pred = test_x @ w_p
            preds.append(pred[:, 0, 0])

        return torch.stack(preds, dim=1)


# Lasso regression (for sparse linear regression).
# Seems to take more time as we decrease alpha.
class LassoModel:
    def __init__(self, alpha, max_iter=100000):
        # the l1 regularizer gets multiplied by alpha.
        self.alpha = alpha
        self.max_iter = max_iter
        self.name = f"lasso_alpha={alpha}_max_iter={max_iter}"

    # inds is a list containing indices where we want the prediction.
    # prediction made at all indices by default.
    def __call__(self, xs, ys, inds=None):
        xs, ys = xs.cpu(), ys.cpu()

        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []  # predict one for first point

        # i: loop over num_points
        # j: loop over bsize
        for i in inds:
            pred = torch.zeros_like(ys[:, 0])

            if i > 0:
                pred = torch.zeros_like(ys[:, 0])
                for j in range(ys.shape[0]):
                    train_xs, train_ys = xs[j, :i], ys[j, :i]

                    # If all points till now have the same label, predict that label.

                    clf = Lasso(
                        alpha=self.alpha, fit_intercept=False, max_iter=self.max_iter
                    )

                    # Check for convergence.
                    with warnings.catch_warnings():
                        warnings.filterwarnings("error")
                        try:
                            clf.fit(train_xs, train_ys)
                        except Warning:
                            print(f"lasso convergence warning at i={i}, j={j}.")
                            raise

                    w_pred = torch.from_numpy(clf.coef_).unsqueeze(1)

                    test_x = xs[j, i : i + 1]
                    y_pred = (test_x @ w_pred.float()).squeeze(1)
                    pred[j] = y_pred[0]

            preds.append(pred)

        return torch.stack(preds, dim=1)


# Gradient Descent and variants.
# Example usage: gd_model = GDModel(NeuralNetwork, {'in_size': 50, 'hidden_size':400, 'out_size' :1}, opt_alg = 'adam', batch_size = 100, lr = 5e-3, num_steps = 200)
class GDModel:
    def __init__(
        self,
        model_class,
        model_class_args,
        opt_alg="sgd",
        batch_size=1,
        num_steps=1000,
        lr=1e-3,
        loss_name="squared",
    ):
        # model_class: torch.nn model class
        # model_class_args: a dict containing arguments for model_class
        # opt_alg can be 'sgd' or 'adam'
        # verbose: whether to print the progress or not
        # batch_size: batch size for sgd
        self.model_class = model_class
        self.model_class_args = model_class_args
        self.opt_alg = opt_alg
        self.lr = lr
        self.batch_size = batch_size
        self.num_steps = num_steps
        self.loss_name = loss_name

        self.name = f"gd_model_class={model_class}_model_class_args={model_class_args}_opt_alg={opt_alg}_lr={lr}_batch_size={batch_size}_num_steps={num_steps}_loss_name={loss_name}"

    def __call__(self, xs, ys, inds=None, verbose=False, print_step=100):
        # inds is a list containing indices where we want the prediction.
        # prediction made at all indices by default.
        # xs: bsize X npoints X ndim.
        # ys: bsize X npoints.
        xs, ys = xs.to(device), ys.to(device)

        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []  # predict one for first point

        # i: loop over num_points
        for i in tqdm(inds):
            pred = torch.zeros_like(ys[:, 0])
            model = ParallelNetworks(
                ys.shape[0], self.model_class, **self.model_class_args
            )
            model.to(device)
            if i > 0:
                pred = torch.zeros_like(ys[:, 0])

                train_xs, train_ys = xs[:, :i], ys[:, :i]
                test_xs, test_ys = xs[:, i : i + 1], ys[:, i : i + 1]

                if self.opt_alg == "sgd":
                    optimizer = torch.optim.SGD(model.parameters(), lr=self.lr)
                elif self.opt_alg == "adam":
                    optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
                else:
                    raise NotImplementedError(f"{self.opt_alg} not implemented.")

                if self.loss_name == "squared":
                    loss_criterion = torch.nn.MSELoss()
                else:
                    raise NotImplementedError(f"{self.loss_name} not implemented.")

                # Training loop
                for j in range(self.num_steps):

                    # Prepare batch
                    mask = torch.zeros(i).bool()
                    perm = torch.randperm(i)
                    mask[perm[: self.batch_size]] = True
                    train_xs_cur, train_ys_cur = train_xs[:, mask, :], train_ys[:, mask]

                    if verbose and j % print_step == 0:
                        model.eval()
                        with torch.no_grad():
                            outputs = model(train_xs_cur)
                            loss = loss_criterion(
                                outputs[:, :, 0], train_ys_cur
                            ).detach()
                            outputs_test = model(test_xs)
                            test_loss = loss_criterion(
                                outputs_test[:, :, 0], test_ys
                            ).detach()
                            print(
                                f"ind:{i},step:{j}, train_loss:{loss.item()}, test_loss:{test_loss.item()}"
                            )

                    optimizer.zero_grad()

                    model.train()
                    outputs = model(train_xs_cur)
                    loss = loss_criterion(outputs[:, :, 0], train_ys_cur)
                    loss.backward()
                    optimizer.step()

                model.eval()
                pred = model(test_xs).detach()

                assert pred.shape[1] == 1 and pred.shape[2] == 1
                pred = pred[:, 0, 0]

            preds.append(pred)

        return torch.stack(preds, dim=1)


class DecisionTreeModel:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.name = f"decision_tree_max_depth={max_depth}"

    # inds is a list containing indices where we want the prediction.
    # prediction made at all indices by default.
    def __call__(self, xs, ys, inds=None):
        xs, ys = xs.cpu(), ys.cpu()

        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []

        # i: loop over num_points
        # j: loop over bsize
        for i in inds:
            pred = torch.zeros_like(ys[:, 0])

            if i > 0:
                pred = torch.zeros_like(ys[:, 0])
                for j in range(ys.shape[0]):
                    train_xs, train_ys = xs[j, :i], ys[j, :i]

                    clf = tree.DecisionTreeRegressor(max_depth=self.max_depth)
                    clf = clf.fit(train_xs, train_ys)
                    test_x = xs[j, i : i + 1]
                    y_pred = clf.predict(test_x)
                    pred[j] = y_pred[0]

            preds.append(pred)

        return torch.stack(preds, dim=1)


class XGBoostModel:
    def __init__(self):
        self.name = "xgboost"

    # inds is a list containing indices where we want the prediction.
    # prediction made at all indices by default.
    def __call__(self, xs, ys, inds=None):
        xs, ys = xs.cpu(), ys.cpu()

        if inds is None:
            inds = range(ys.shape[1])
        else:
            if max(inds) >= ys.shape[1] or min(inds) < 0:
                raise ValueError("inds contain indices where xs and ys are not defined")

        preds = []

        # i: loop over num_points
        # j: loop over bsize
        for i in tqdm(inds):
            pred = torch.zeros_like(ys[:, 0])
            if i > 0:
                pred = torch.zeros_like(ys[:, 0])
                for j in range(ys.shape[0]):
                    train_xs, train_ys = xs[j, :i], ys[j, :i]

                    clf = xgb.XGBRegressor()

                    clf = clf.fit(train_xs, train_ys)
                    test_x = xs[j, i : i + 1]
                    y_pred = clf.predict(test_x)
                    pred[j] = y_pred[0].item()

            preds.append(pred)

        return torch.stack(preds, dim=1)

In [6]:
# Curriculum
# Implementation of curriculum learning for the models

import math

class Curriculum:
    def __init__(self, args):
        # args.dims and args.points each contain start, end, inc, interval attributes
        # inc denotes the change in n_dims,
        # this change is done every interval,
        # and start/end are the limits of the parameter
        self.n_dims_truncated = args.dims.start
        self.n_points = args.points.start
        self.n_dims_schedule = args.dims
        self.n_points_schedule = args.points
        self.step_count = 0

    def update(self):
        self.step_count += 1
        self.n_dims_truncated = self.update_var(
            self.n_dims_truncated, self.n_dims_schedule
        )
        self.n_points = self.update_var(self.n_points, self.n_points_schedule)

    def update_var(self, var, schedule):
        if self.step_count % schedule.interval == 0:
            var += schedule.inc

        return min(var, schedule.end)


# returns the final value of var after applying curriculum.
def get_final_var(init_var, total_steps, inc, n_steps, lim):
    final_var = init_var + math.floor((total_steps) / n_steps) * inc

    return min(final_var, lim)

In [27]:
# Samplers
# Data sampling functionality for training and evaluation

import math
import torch

class DataSampler:
    def __init__(self, n_dims):
        self.n_dims = n_dims

    def sample_xs(self):
        raise NotImplementedError


def get_data_sampler(data_name, n_dims, **kwargs):
    names_to_classes = {
        "gaussian": GaussianSampler,
    }
    if data_name in names_to_classes:
        sampler_cls = names_to_classes[data_name]
        return sampler_cls(n_dims, **kwargs)
    else:
        print("Unknown sampler")
        raise NotImplementedError


def sample_transformation(eigenvalues, normalize=False):
    n_dims = len(eigenvalues)
    U, _, _ = torch.linalg.svd(torch.randn(n_dims, n_dims))
    t = U @ torch.diag(eigenvalues) @ torch.transpose(U, 0, 1)
    if normalize:
        norm_subspace = torch.sum(eigenvalues**2)
        t *= math.sqrt(n_dims / norm_subspace)
    return t


class GaussianSampler(DataSampler):
    def __init__(self, n_dims, bias=None, scale=None):
        super().__init__(n_dims)
        self.bias = bias
        self.scale = scale

    def sample_xs(self, n_points, b_size, n_dims_truncated=None, seeds=None):
        if seeds is None:
            xs_b = torch.randn(b_size, n_points, self.n_dims)
        else:
            xs_b = torch.zeros(b_size, n_points, self.n_dims)
            generator = torch.Generator()
            assert len(seeds) == b_size
            for i, seed in enumerate(seeds):
                generator.manual_seed(seed)
                xs_b[i] = torch.randn(n_points, self.n_dims, generator=generator)
        if self.scale is not None:
            xs_b = xs_b @ self.scale
        if self.bias is not None:
            xs_b += self.bias
        if n_dims_truncated is not None:
            xs_b[:, :, n_dims_truncated:] = 0
        return xs_b

In [28]:
# Tasks
# Task definitions for the in-context learning experiments

import math
import torch


def squared_error(ys_pred, ys):
    return (ys - ys_pred).square()


def mean_squared_error(ys_pred, ys):
    return (ys - ys_pred).square().mean()


def accuracy(ys_pred, ys):
    return (ys == ys_pred.sign()).float()


sigmoid = torch.nn.Sigmoid()
bce_loss = torch.nn.BCELoss()


def cross_entropy(ys_pred, ys):
    output = sigmoid(ys_pred)
    target = (ys + 1) / 2
    return bce_loss(output, target)


class Task:
    def __init__(self, n_dims, batch_size, pool_dict=None, seeds=None):
        self.n_dims = n_dims
        self.b_size = batch_size
        self.pool_dict = pool_dict
        self.seeds = seeds
        assert pool_dict is None or seeds is None

    def evaluate(self, xs):
        raise NotImplementedError

    @staticmethod
    def generate_pool_dict(n_dims, num_tasks):
        raise NotImplementedError

    @staticmethod
    def get_metric():
        raise NotImplementedError

    @staticmethod
    def get_training_metric():
        raise NotImplementedError


def get_task_sampler(
    task_name, n_dims, batch_size, pool_dict=None, num_tasks=None, **kwargs
):
    task_names_to_classes = {
        "linear_regression": LinearRegression,
        "sparse_linear_regression": SparseLinearRegression,
        "linear_classification": LinearClassification,
        "noisy_linear_regression": NoisyLinearRegression,
        "quadratic_regression": QuadraticRegression,
        "relu_2nn_regression": Relu2nnRegression,
        "decision_tree": DecisionTree,
    }
    if task_name in task_names_to_classes:
        task_cls = task_names_to_classes[task_name]
        if num_tasks is not None:
            if pool_dict is not None:
                raise ValueError("Either pool_dict or num_tasks should be None.")
            pool_dict = task_cls.generate_pool_dict(n_dims, num_tasks, **kwargs)
        return lambda **args: task_cls(n_dims, batch_size, pool_dict, **args, **kwargs)
    else:
        print("Unknown task")
        raise NotImplementedError


class LinearRegression(Task):
    def __init__(self, n_dims, batch_size, pool_dict=None, seeds=None, scale=1):
        """scale: a constant by which to scale the randomly sampled weights."""
        super(LinearRegression, self).__init__(n_dims, batch_size, pool_dict, seeds)
        self.scale = scale

        if pool_dict is None and seeds is None:
            self.w_b = torch.randn(self.b_size, self.n_dims, 1)
        elif seeds is not None:
            self.w_b = torch.zeros(self.b_size, self.n_dims, 1)
            generator = torch.Generator()
            assert len(seeds) == self.b_size
            for i, seed in enumerate(seeds):
                generator.manual_seed(seed)
                self.w_b[i] = torch.randn(self.n_dims, 1, generator=generator)
        else:
            assert "w" in pool_dict
            indices = torch.randperm(len(pool_dict["w"]))[:batch_size]
            self.w_b = pool_dict["w"][indices]

    def evaluate(self, xs_b):
        w_b = self.w_b.to(xs_b.device)
        ys_b = self.scale * (xs_b @ w_b)[:, :, 0]
        return ys_b

    @staticmethod
    def generate_pool_dict(n_dims, num_tasks, **kwargs):  # ignore extra args
        return {"w": torch.randn(num_tasks, n_dims, 1)}

    @staticmethod
    def get_metric():
        return squared_error

    @staticmethod
    def get_training_metric():
        return mean_squared_error


class SparseLinearRegression(LinearRegression):
    def __init__(
        self,
        n_dims,
        batch_size,
        pool_dict=None,
        seeds=None,
        scale=1,
        sparsity=3,
        valid_coords=None,
    ):
        """scale: a constant by which to scale the randomly sampled weights."""
        super(SparseLinearRegression, self).__init__(
            n_dims, batch_size, pool_dict, seeds, scale
        )
        self.sparsity = sparsity
        if valid_coords is None:
            valid_coords = n_dims
        assert valid_coords <= n_dims

        for i, w in enumerate(self.w_b):
            mask = torch.ones(n_dims).bool()
            if seeds is None:
                perm = torch.randperm(valid_coords)
            else:
                generator = torch.Generator()
                generator.manual_seed(seeds[i])
                perm = torch.randperm(valid_coords, generator=generator)
            mask[perm[:sparsity]] = False
            w[mask] = 0

    def evaluate(self, xs_b):
        w_b = self.w_b.to(xs_b.device)
        ys_b = self.scale * (xs_b @ w_b)[:, :, 0]
        return ys_b

    @staticmethod
    def get_metric():
        return squared_error

    @staticmethod
    def get_training_metric():
        return mean_squared_error


class LinearClassification(LinearRegression):
    def evaluate(self, xs_b):
        ys_b = super().evaluate(xs_b)
        return ys_b.sign()

    @staticmethod
    def get_metric():
        return accuracy

    @staticmethod
    def get_training_metric():
        return cross_entropy


class NoisyLinearRegression(LinearRegression):
    def __init__(
        self,
        n_dims,
        batch_size,
        pool_dict=None,
        seeds=None,
        scale=1,
        noise_std=0,
        renormalize_ys=False,
    ):
        """noise_std: standard deviation of noise added to the prediction."""
        super(NoisyLinearRegression, self).__init__(
            n_dims, batch_size, pool_dict, seeds, scale
        )
        self.noise_std = noise_std
        self.renormalize_ys = renormalize_ys

    def evaluate(self, xs_b):
        ys_b = super().evaluate(xs_b)
        ys_b_noisy = ys_b + torch.randn_like(ys_b) * self.noise_std
        if self.renormalize_ys:
            ys_b_noisy = ys_b_noisy * math.sqrt(self.n_dims) / ys_b_noisy.std()

        return ys_b_noisy


class QuadraticRegression(LinearRegression):
    def evaluate(self, xs_b):
        w_b = self.w_b.to(xs_b.device)
        ys_b_quad = ((xs_b**2) @ w_b)[:, :, 0]
        #         ys_b_quad = ys_b_quad * math.sqrt(self.n_dims) / ys_b_quad.std()
        # Renormalize to Linear Regression Scale
        ys_b_quad = ys_b_quad / math.sqrt(3)
        ys_b_quad = self.scale * ys_b_quad
        return ys_b_quad


class Relu2nnRegression(Task):
    def __init__(
        self,
        n_dims,
        batch_size,
        pool_dict=None,
        seeds=None,
        scale=1,
        hidden_layer_size=100,
    ):
        """scale: a constant by which to scale the randomly sampled weights."""
        super(Relu2nnRegression, self).__init__(n_dims, batch_size, pool_dict, seeds)
        self.scale = scale
        self.hidden_layer_size = hidden_layer_size

        if pool_dict is None and seeds is None:
            self.W1 = torch.randn(self.b_size, self.n_dims, hidden_layer_size)
            self.W2 = torch.randn(self.b_size, hidden_layer_size, 1)
        elif seeds is not None:
            self.W1 = torch.zeros(self.b_size, self.n_dims, hidden_layer_size)
            self.W2 = torch.zeros(self.b_size, hidden_layer_size, 1)
            generator = torch.Generator()
            assert len(seeds) == self.b_size
            for i, seed in enumerate(seeds):
                generator.manual_seed(seed)
                self.W1[i] = torch.randn(
                    self.n_dims, hidden_layer_size, generator=generator
                )
                self.W2[i] = torch.randn(hidden_layer_size, 1, generator=generator)
        else:
            assert "W1" in pool_dict and "W2" in pool_dict
            assert len(pool_dict["W1"]) == len(pool_dict["W2"])
            indices = torch.randperm(len(pool_dict["W1"]))[:batch_size]
            self.W1 = pool_dict["W1"][indices]
            self.W2 = pool_dict["W2"][indices]

    def evaluate(self, xs_b):
        W1 = self.W1.to(xs_b.device)
        W2 = self.W2.to(xs_b.device)
        # Renormalize to Linear Regression Scale
        ys_b_nn = (torch.nn.functional.relu(xs_b @ W1) @ W2)[:, :, 0]
        ys_b_nn = ys_b_nn * math.sqrt(2 / self.hidden_layer_size)
        ys_b_nn = self.scale * ys_b_nn
        #         ys_b_nn = ys_b_nn * math.sqrt(self.n_dims) / ys_b_nn.std()
        return ys_b_nn

    @staticmethod
    def generate_pool_dict(n_dims, num_tasks, hidden_layer_size=4, **kwargs):
        return {
            "W1": torch.randn(num_tasks, n_dims, hidden_layer_size),
            "W2": torch.randn(num_tasks, hidden_layer_size, 1),
        }

    @staticmethod
    def get_metric():
        return squared_error

    @staticmethod
    def get_training_metric():
        return mean_squared_error


class DecisionTree(Task):
    def __init__(self, n_dims, batch_size, pool_dict=None, seeds=None, depth=4):

        super(DecisionTree, self).__init__(n_dims, batch_size, pool_dict, seeds)
        self.depth = depth

        if pool_dict is None:

            # We represent the tree using an array (tensor). Root node is at index 0, its 2 children at index 1 and 2...
            # dt_tensor stores the coordinate used at each node of the decision tree.
            # Only indices corresponding to non-leaf nodes are relevant
            self.dt_tensor = torch.randint(
                low=0, high=n_dims, size=(batch_size, 2 ** (depth + 1) - 1)
            )

            # Target value at the leaf nodes.
            # Only indices corresponding to leaf nodes are relevant.
            self.target_tensor = torch.randn(self.dt_tensor.shape)
        elif seeds is not None:
            self.dt_tensor = torch.zeros(batch_size, 2 ** (depth + 1) - 1)
            self.target_tensor = torch.zeros_like(self.dt_tensor)
            generator = torch.Generator()
            assert len(seeds) == self.b_size
            for i, seed in enumerate(seeds):
                generator.manual_seed(seed)
                self.dt_tensor[i] = torch.randint(
                    low=0,
                    high=n_dims - 1,
                    size=2 ** (depth + 1) - 1,
                    generator=generator,
                )
                self.target_tensor[i] = torch.randn(
                    self.dt_tensor[i].shape, generator=generator
                )
        else:
            raise NotImplementedError

    def evaluate(self, xs_b):
        dt_tensor = self.dt_tensor.to(xs_b.device)
        target_tensor = self.target_tensor.to(xs_b.device)
        ys_b = torch.zeros(xs_b.shape[0], xs_b.shape[1], device=xs_b.device)
        for i in range(xs_b.shape[0]):
            xs_bool = xs_b[i] > 0
            # If a single decision tree present, use it for all the xs in the batch.
            if self.b_size == 1:
                dt = dt_tensor[0]
                target = target_tensor[0]
            else:
                dt = dt_tensor[i]
                target = target_tensor[i]

            cur_nodes = torch.zeros(xs_b.shape[1], device=xs_b.device).long()
            for j in range(self.depth):
                cur_coords = dt[cur_nodes]
                cur_decisions = xs_bool[torch.arange(xs_bool.shape[0]), cur_coords]
                cur_nodes = 2 * cur_nodes + 1 + cur_decisions

            ys_b[i] = target[cur_nodes]

        return ys_b

    @staticmethod
    def generate_pool_dict(n_dims, num_tasks, hidden_layer_size=4, **kwargs):
        raise NotImplementedError

    @staticmethod
    def get_metric():
        return squared_error

    @staticmethod
    def get_training_metric():
        return mean_squared_error

In [29]:
import os
import uuid # Added missing import

# Configuration
# Define all configuration variables in this cell

# Class to simplify dict access
class Config(dict):
    def __init__(self, *args, **kwargs):
        super(Config, self).__init__(*args, **kwargs)
        self.__dict__ = self # This maps attribute access to dict key access

    # __getattr__ handles attribute access for keys *not* already in __dict__
    # However, since __dict__ is set to self, this is mostly redundant
    # unless keys are added *after* __init__ without updating __dict__
    # Keeping it doesn't hurt, but it's often not strictly needed with __dict__ = self
    def __getattr__(self, name):
        if name in self:
            return self[name]
        # It's better practice to raise AttributeError if the key doesn't exist
        raise AttributeError(f"No such attribute: {name}")

    # __setattr__ ensures that setting an attribute also updates the dictionary
    def __setattr__(self, name, value):
        self[name] = value
        # Keep __dict__ consistent if necessary, although direct assignment is usually enough
        self.__dict__[name] = value

    def get_nested(self, path):
        """Get a nested attribute using dot notation (e.g., 'training.curriculum.dims.start')"""
        parts = path.split('.')
        curr = self
        for part in parts:
            # Handle both Config objects (attribute access) and dicts (key access)
            try:
                if isinstance(curr, Config) or isinstance(curr, dict):
                     # Check if it's a Config object first for attribute access preference
                     if hasattr(curr, part):
                         curr = getattr(curr, part)
                     elif part in curr: # Fallback to key access if not an attribute (or for plain dicts)
                         curr = curr[part]
                     else:
                         return None # Part not found
                else:
                    return None # Current item is not a dictionary or Config object
            except (AttributeError, KeyError):
                 return None
        return curr

# Model Configuration
model_config = {
    "family": "gpt2",
    "n_dims": 20,
    "n_positions": 101,
    "n_embd": 128,
    "n_layer": 12,
    "n_head": 4
}

# Curriculum Configuration
dims_curriculum = {
    "start": 5,
    "end": 20,
    "inc": 1,
    "interval": 2000
}

points_curriculum = {
    "start": 26,
    "end": 101,
    "inc": 5,
    "interval": 2000
}

# --- FIX START ---
# Wrap the nested dictionaries manually
dims_curriculum_cfg = Config(dims_curriculum)
points_curriculum_cfg = Config(points_curriculum)

curriculum_config = {
    "dims": dims_curriculum_cfg, # Use the Config objects here
    "points": points_curriculum_cfg # Use the Config objects here
}
# --- FIX END ---


# Training Configuration
training_config = {
    "task": "relu_2nn_regression",
    "task_kwargs": {"hidden_layer_size": 100},
    "num_tasks": None,
    "num_training_examples": None,
    "data": "gaussian",
    "batch_size": 64,
    "learning_rate": 0.0001,
    "train_steps": 50000,  # Reduced for demonstration purposes
    "save_every_steps": 1000,
    "keep_every_steps": 10000,
    "resume_id": None,
    # "curriculum": curriculum_config # This will be set later in the combined config
}

# WandB Configuration
wandb_config = {
    "project": "in-context-training",
    "entity": "in-context",
    "notes": "",
    "name": "relu_2nn_regression_standard",
    "log_every_steps": 10
}

# Output Directory Configuration
output_config = {
    "base_dir": "./models/",
    "run_dir": None  # Will be set during execution
}

# Test Run Flag
test_run = False

# Use WandB Flag
use_wandb = False # Assuming you might want this flag later

# Create config objects for each section
model_cfg = Config(model_config)
# curriculum_cfg is created using the pre-wrapped nested configs
curriculum_cfg = Config(curriculum_config)
training_cfg = Config(training_config)
# Add the curriculum config to the training config *after* both are Config objects
training_cfg.curriculum = curriculum_cfg # Now training_cfg.curriculum points to the curriculum_cfg object
wandb_cfg = Config(wandb_config)
output_cfg = Config(output_config)

# Combined config for convenience
config = Config({
    "model": model_cfg,
    "training": training_cfg,
    "wandb": wandb_cfg,
    "output": output_cfg,
    "test_run": test_run,
    "use_wandb": use_wandb # Added flag to main config
})

# Configure output directory
# Ensure base directory exists
os.makedirs(output_config["base_dir"], exist_ok=True) # Use original dict key here or output_cfg.base_dir

# Use run_dir from the Config object now
output_cfg.run_dir = os.path.join(output_cfg.base_dir, training_cfg.task)
os.makedirs(output_cfg.run_dir, exist_ok=True) # Create task directory

if not config.test_run: # Access test_run from the main config object
    # Generate a unique run ID if not resuming
    if training_cfg.resume_id is None:
        run_id = str(uuid.uuid4())[:8]
    else:
        run_id = training_cfg.resume_id
    output_cfg.run_dir = os.path.join(output_cfg.run_dir, run_id)
    os.makedirs(output_cfg.run_dir, exist_ok=True) # Create specific run directory

# Print configuration
print(f"Model: {config.model.family} with {config.model.n_layer} layers, {config.model.n_head} heads")
print(f"Task: {config.training.task} with {config.training.task_kwargs}")
# Access curriculum through the main config object and its nested Config objects
print(f"Curriculum: {config.training.curriculum.dims.start}-{config.training.curriculum.dims.end} dims, {config.training.curriculum.points.start}-{config.training.curriculum.points.end} points")
print(f"Test run: {config.test_run}")
print(f"Output directory: {config.output.run_dir}")
print(f"Using WandB: {config.use_wandb}")

# Example of using get_nested
print(f"Nested access check (dims start): {config.get_nested('training.curriculum.dims.start')}")

Model: gpt2 with 12 layers, 4 heads
Task: relu_2nn_regression with {'hidden_layer_size': 100}
Curriculum: 5-20 dims, 26-101 points
Test run: False
Output directory: ./models/relu_2nn_regression/458e06b0
Using WandB: False
Nested access check (dims start): 5


In [30]:
# Define Training Functions
def train_step(model, xs, ys, optimizer, loss_func):
    optimizer.zero_grad()
    output = model(xs, ys)
    loss = loss_func(output, ys)
    loss.backward()
    optimizer.step()
    return loss.detach().item(), output.detach()


def sample_seeds(total_seeds, count):
    seeds = set()
    while len(seeds) < count:
        seeds.add(randint(0, total_seeds - 1))
    return seeds

In [31]:
# Define the Train Function
from tqdm.notebook import tqdm
def train(model, config, use_wandb=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=config.training.learning_rate)
    curriculum = Curriculum(config.training.curriculum)

    # Setup for tracking metrics
    metrics_history = {
        'overall_loss': [],
        'excess_loss': [],
        'n_points': [],
        'n_dims': []
    }

    starting_step = 0
    state_path = os.path.join(config.output.run_dir, "state.pt")
    if os.path.exists(state_path):
        state = torch.load(state_path)
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        starting_step = state["train_step"]
        for i in range(state["train_step"] + 1):
            curriculum.update()

    n_dims = model.n_dims
    bsize = config.training.batch_size
    data_sampler = get_data_sampler(config.training.data, n_dims=n_dims)
    task_sampler = get_task_sampler(
        config.training.task,
        n_dims,
        bsize,
        num_tasks=config.training.num_tasks,
        **config.training.task_kwargs,
    )
    pbar = tqdm(range(starting_step, config.training.train_steps))

    num_training_examples = config.training.num_training_examples

    for i in pbar:
        data_sampler_args = {}
        task_sampler_args = {}

        if "sparse" in config.training.task:
            task_sampler_args["valid_coords"] = curriculum.n_dims_truncated
        if num_training_examples is not None:
            assert num_training_examples >= bsize
            seeds = sample_seeds(num_training_examples, bsize)
            data_sampler_args["seeds"] = seeds
            task_sampler_args["seeds"] = [s + 1 for s in seeds]

        xs = data_sampler.sample_xs(
            curriculum.n_points,
            bsize,
            curriculum.n_dims_truncated,
            **data_sampler_args,
        )
        task = task_sampler(**task_sampler_args)
        ys = task.evaluate(xs)

        loss_func = task.get_training_metric()

        # Use the device variable instead of hardcoded cuda()
        loss, output = train_step(model, xs.to(device), ys.to(device), optimizer, loss_func)

        point_wise_tags = list(range(curriculum.n_points))
        point_wise_loss_func = task.get_metric()
        point_wise_loss = point_wise_loss_func(output, ys.to(device)).mean(dim=0)

        baseline_loss = (
            sum(
                max(curriculum.n_dims_truncated - ii, 0)
                for ii in range(curriculum.n_points)
            )
            / curriculum.n_points
        )

        # Track metrics for plotting
        metrics_history['overall_loss'].append(loss)
        metrics_history['excess_loss'].append(loss / baseline_loss)
        metrics_history['n_points'].append(curriculum.n_points)
        metrics_history['n_dims'].append(curriculum.n_dims_truncated)

        if i % config.wandb.log_every_steps == 0 and use_wandb and not config.test_run:
            import wandb
            wandb.log(
                {
                    "overall_loss": loss,
                    "excess_loss": loss / baseline_loss,
                    "pointwise/loss": dict(
                        zip(point_wise_tags, point_wise_loss.cpu().numpy())
                    ),
                    "n_points": curriculum.n_points,
                    "n_dims": curriculum.n_dims_truncated,
                },
                step=i,
            )

        curriculum.update()

        pbar.set_description(f"loss {loss:.4f}")
        if i % config.training.save_every_steps == 0 and not config.test_run:
            # Make sure output directory exists
            os.makedirs(config.output.run_dir, exist_ok=True)
            training_state = {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_step": i,
            }
            torch.save(training_state, state_path)

        if (
            config.training.keep_every_steps > 0
            and i % config.training.keep_every_steps == 0
            and not config.test_run
            and i > 0
        ):
            # Make sure output directory exists
            os.makedirs(config.output.run_dir, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(config.output.run_dir, f"model_{i}.pt"))

    return metrics_history

In [32]:
# Test Configuration
# You can modify these parameters to customize your training

# For running quick tests
test_mode = {
    # Set to True for a quick test run with minimal iterations
    "enabled": False,
    # Model size reduction for faster testing
    "reduced_model": {
        "n_layer": 3,
        "n_head": 2
    },
    # Reduced training steps
    "train_steps": 100
}

# Task Configuration
task_options = {
    # Available tasks: linear_regression, sparse_linear_regression, relu_2nn_regression, decision_tree
    "task_name": "relu_2nn_regression",
    "task_kwargs": {"hidden_layer_size": 100}
}

# Apply custom configuration if needed
if test_mode["enabled"]:
    config.test_run = True
    config.model.n_layer = test_mode["reduced_model"]["n_layer"]
    config.model.n_head = test_mode["reduced_model"]["n_head"]
    config.training.train_steps = test_mode["train_steps"]
    # Use the full range for curriculum in test mode
    config.training.curriculum.dims.start = config.training.curriculum.dims.end
    config.training.curriculum.points.start = config.training.curriculum.points.end
    print("TEST MODE ENABLED - Running with reduced settings")

# Update task if specified
config.training.task = task_options["task_name"]
config.training.task_kwargs = task_options["task_kwargs"]

# Update output directory based on task
config.output.run_dir = os.path.join(config.output.base_dir, config.training.task)

# Print the final configuration
print(f"Task: {config.training.task}")
print(f"Model: {config.model.n_layer} layers, {config.model.n_head} heads")
print(f"Training steps: {config.training.train_steps}")
print(f"Test mode: {config.test_run}")
print(f"Output directory: {config.output.run_dir}")

Task: relu_2nn_regression
Model: 12 layers, 4 heads
Training steps: 50000
Test mode: False
Output directory: ./models/relu_2nn_regression


In [33]:
# Build model
model = build_model(config.model)
model.to(device)  # Use device variable instead of hardcoded cuda()
model.train()

# Print model summary
print(f"Model: {model.name}")
print(f"Task: {config.training.task}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Using device: {device}")

# Adjust curriculum if it's a test run
if config.test_run:
    config.training.curriculum.dims.start = config.training.curriculum.dims.end
    config.training.curriculum.points.start = config.training.curriculum.points.end

# Initialize WandB if enabled
if use_wandb and not config.test_run:
    try:
        import wandb
        # Initialize wandb
        wandb.init(
            dir=config.output.run_dir,
            project=config.wandb.project,
            entity=config.wandb.entity,
            config=config,
            notes=config.wandb.notes,
            name=config.wandb.name,
            resume=True,
        )
        wandb_initialized = True
        print("WandB initialized successfully")
    except ImportError:
        print("WandB not installed, continuing without it")
        use_wandb = False
        wandb_initialized = False
else:
    wandb_initialized = False

# Train the model
metrics_history = train(model, config, use_wandb=wandb_initialized)

Model: gpt2_embd=128_layer=12_head=4
Task: relu_2nn_regression
Number of parameters: 8,841,089
Using device: cuda


  0%|          | 0/49000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Build Model and Train

In [34]:
# Build model
model = build_model(config.model)
model.to(device)  # Use device variable instead of hardcoded cuda()
model.train()

# Print model summary
print(f"Model: {model.name}")
print(f"Task: {config.training.task}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Using device: {device}")

# Adjust curriculum if it's a test run
if config.test_run:
    config.training.curriculum.dims.start = config.training.curriculum.dims.end
    config.training.curriculum.points.start = config.training.curriculum.points.end

# Train the model
metrics_history = train(model, config, use_wandb=use_wandb)

Model: gpt2_embd=128_layer=12_head=4
Task: relu_2nn_regression
Number of parameters: 8,841,089
Using device: cuda


  0%|          | 0/49000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Visualize Training Progress

In [35]:
# Plot training curves if metrics_history is not empty
if metrics_history and len(metrics_history['overall_loss']) > 0:
    plt.figure(figsize=(15, 10))

    # Plot loss
    plt.subplot(2, 2, 1)
    plt.plot(metrics_history['overall_loss'])
    plt.title('Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)

    # Plot excess loss
    plt.subplot(2, 2, 2)
    plt.plot(metrics_history['excess_loss'])
    plt.title('Excess Loss (Loss / Baseline Loss)')
    plt.xlabel('Step')
    plt.ylabel('Excess Loss')
    plt.yscale('log')
    plt.grid(True)

    # Plot number of points
    plt.subplot(2, 2, 3)
    plt.plot(metrics_history['n_points'])
    plt.title('Number of Points in Curriculum')
    plt.xlabel('Step')
    plt.ylabel('Number of Points')
    plt.grid(True)

    # Plot number of dimensions
    plt.subplot(2, 2, 4)
    plt.plot(metrics_history['n_dims'])
    plt.title('Number of Dimensions in Curriculum')
    plt.xlabel('Step')
    plt.ylabel('Number of Dimensions')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    # Also print the final metrics
    print(f"Final metrics after {len(metrics_history['overall_loss'])} steps:")
    print(f"  Loss: {metrics_history['overall_loss'][-1]:.6f}")
    print(f"  Excess Loss: {metrics_history['excess_loss'][-1]:.6f}")
    print(f"  Points: {metrics_history['n_points'][-1]}")
    print(f"  Dimensions: {metrics_history['n_dims'][-1]}")

    # Save the plot if not in test mode
    if not config.test_run:
        try:
            plot_path = os.path.join(config.output.run_dir, "training_metrics.png")
            plt.savefig(plot_path)
            print(f"Plot saved to {plot_path}")
        except Exception as e:
            print(f"Error saving plot: {e}")
else:
    print("No training metrics available to plot.")

No training metrics available to plot.


## Evaluate the Model (Optional)

We can use the evaluation code to assess model performance.

In [36]:
# Skip evaluation for test runs
if not config.test_run:
    try:
        # Run evaluation
        print("Running evaluation...")
        metrics = get_run_metrics(config.out_dir)

        # Print summary of metrics
        print("\nEvaluation metrics:")
        for eval_name, models_metrics in metrics.items():
            print(f"\n{eval_name}:")
            if "mean" in models_metrics:
                last_mean = models_metrics["mean"][-1]
                print(f"  Final mean: {last_mean:.4f}")
            else:
                for model_name, model_metrics in models_metrics.items():
                    print(f"  {model_name}:")
                    if "mean" in model_metrics:
                        last_mean = model_metrics["mean"][-1]
                        print(f"    Final mean: {last_mean:.4f}")
    except Exception as e:
        print(f"Evaluation failed: {e}")
        print("Continuing without evaluation.")

Running evaluation...
Evaluation failed: name 'get_run_metrics' is not defined
Continuing without evaluation.


## Save the Final Model

In [39]:
# Save the final model
import yaml

if not config.test_run:
    # Create directory if it doesn't exist
    os.makedirs(config.output.run_dir, exist_ok=True)

    # Save the model state dict
    final_model_path = os.path.join(config.output.run_dir, "final_model.pt")
    torch.save(model.state_dict(), final_model_path)
    print(f"Final model saved to {final_model_path}")

    # Also save the full model (architecture + weights) for easier loading
    full_model_path = os.path.join(config.output.run_dir, "full_model.pt")
    torch.save(model, full_model_path)
    print(f"Full model saved to {full_model_path}")

    # Save the config as YAML
    config_path = os.path.join(config.output.run_dir, "config.yaml")
    with open(config_path, "w") as yaml_file:
        # Convert config to plain dictionary for YAML serialization
        config_dict = {
            "model": dict(config.model),
            "training": dict(config.training),
            "wandb": dict(config.wandb),
            "output": dict(config.output),
            "test_run": config.test_run
        }
        yaml.dump(config_dict, yaml_file, default_flow_style=False)
    print(f"Configuration saved to {config_path}")
else:
    print("Test run completed. Model not saved.")

Final model saved to ./models/relu_2nn_regression/final_model.pt
Full model saved to ./models/relu_2nn_regression/full_model.pt
Configuration saved to ./models/relu_2nn_regression/config.yaml


# Evaluate the Model (Optional)
# Skip evaluation for test runs
if not config.test_run:
    try:
        # Run evaluation
        print("Running evaluation...")
        metrics = get_run_metrics(config.output.run_dir)
        
        # Print summary of metrics
        print("\nEvaluation metrics:")
        for eval_name, models_metrics in metrics.items():
            print(f"\n{eval_name}:")
            if "mean" in models_metrics:
                last_mean = models_metrics["mean"][-1]
                print(f"  Final mean: {last_mean:.4f}")
            else:
                for model_name, model_metrics in models_metrics.items():
                    print(f"  {model_name}:")
                    if "mean" in model_metrics:
                        last_mean = model_metrics["mean"][-1]
                        print(f"    Final mean: {last_mean:.4f}")
    except Exception as e:
        print(f"Evaluation failed: {e}")
        print("Continuing without evaluation.")

In [40]:
# Cleanup WandB (if used)
if use_wandb and wandb_initialized and not config.test_run:
    import wandb
    wandb.finish()

print("Training complete!")

# Provide instructions for loading and using the model
print("\nTo load the trained model later:")
print("```python")
print("import torch")
print("from transformers import GPT2Model, GPT2Config")
print("")
print("# Define the TransformerModel class (copy from this notebook)")
print("# ...")
print("")
print("# Option 1: Load just the state dict (requires model definition)")
print("model_config = {'family': 'gpt2', 'n_dims': 20, 'n_positions': 101, 'n_embd': 128, 'n_layer': 12, 'n_head': 4}")
print("model = build_model(model_config)  # Create model with same architecture")
print("model.load_state_dict(torch.load('path/to/final_model.pt'))")
print("")
print("# Option 2: Load the full model (architecture + weights)")
print("model = torch.load('path/to/full_model.pt')")
print("```")

Training complete!

To load the trained model later:
```python
import torch
from transformers import GPT2Model, GPT2Config

# Define the TransformerModel class (copy from this notebook)
# ...

# Option 1: Load just the state dict (requires model definition)
model_config = {'family': 'gpt2', 'n_dims': 20, 'n_positions': 101, 'n_embd': 128, 'n_layer': 12, 'n_head': 4}
model = build_model(model_config)  # Create model with same architecture
model.load_state_dict(torch.load('path/to/final_model.pt'))

# Option 2: Load the full model (architecture + weights)
model = torch.load('path/to/full_model.pt')
```
